In [1]:
import pandas as pd
import re
import nltk
import mlflow
import mlflow.sklearn
from nltk.corpus import stopwords
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Ensure stopwords are available
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

# Preprocessing function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\W", " ", text)
    text = " ".join(word for word in text.split() if word not in stop_words)
    return text

# MLflow setup - Using local file system for tracking
mlflow.set_tracking_uri("file:./mlruns")  
mlflow.set_experiment("Fake News Detection")

def main():
    # Load data
    real_news = pd.read_csv("../data/True.csv")
    fake_news = pd.read_csv("../data/Fake.csv")

    real_news["label"] = 1
    fake_news["label"] = 0

    df = pd.concat([real_news, fake_news]).reset_index(drop=True)
    df["text_clean"] = df["text"].apply(clean_text)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        df["text_clean"], df["label"], test_size=0.2, random_state=42
    )

    # Create pipeline
    model_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, preprocessor=clean_text)),
        ("clf", LogisticRegression())
    ])

    with mlflow.start_run():
        mlflow.sklearn.autolog()

        model_pipeline.fit(X_train, y_train)

        # Predictions
        y_pred = model_pipeline.predict(X_test)

        # Log metrics
        mlflow.log_metrics({
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred)
        })

        # Save model and register it
        mlflow.sklearn.log_model(model_pipeline, "model", registered_model_name="FakeNewsModel")

        print(f"Model logged with run ID: {mlflow.active_run().info.run_id}")

if __name__ == "__main__":
    main()


2025/04/05 15:12:58 WARNING mlflow.utils.autologging_utils: You are using an unsupported version of sklearn. If you encounter errors during autologging, try upgrading / downgrading sklearn to a supported version, or try upgrading MLflow.
2025/04/05 15:12:59 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.
2025/04/05 15:15:09 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.core.series.Series'>. Dataset logging skipped.


Model logged with run ID: e8bffd29639a46ed88dbc25565fc86e3


Registered model 'FakeNewsModel' already exists. Creating a new version of this model...
Created version '3' of model 'FakeNewsModel'.


In [2]:
from mlflow.tracking import MlflowClient

# Initialize the MLflow client
client = MlflowClient()

# Model name and version
model_name = "FakeNewsModel"
model_version = 1  # Version 1

# Set the model stage to 'Production'
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage="Production"
)

print(f"Model {model_name} version {model_version} has been promoted to 'Production' stage.")


C:\Users\Doha\AppData\Local\Temp\ipykernel_42452\2244507630.py:11: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.10.0/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Model FakeNewsModel version 1 has been promoted to 'Production' stage.


In [3]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")  # Make sure this is the same everywhere


In [1]:
import dagshub
dagshub.init(repo_owner='Dohak22', repo_name='latest', mlflow=True)

Accessing as Dohak22

Initialized MLflow to track repo "Dohak22/latest"

Repository Dohak22/latest initialized!

In [2]:
import mlflow
with mlflow.start_run():
  # Your training code here...
  mlflow.log_metric('accuracy', 42)
  mlflow.log_param('Param name', 'Value')

In [3]:
pip install mlflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
